In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

df = pd.read_csv("core_upscaled_dataset_long.csv")
df = df.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates()

# Reconstruct features into a DataFrame to seamlessly handle categorical variables
records = []
for curve_id, group in df.groupby("Curve_ID"):
    group = group.sort_values("Sw")
    
    core_pc = group["Core_Pc"].values.astype(float)
    core_pc_norm = core_pc / core_pc[0]
    
    record = {
        "Curve_ID": curve_id,
        "Core_Porosity": group["Core_Porosity"].iloc[0],
        "Core_Permeability_mD": group["Core_Permeability_mD"].iloc[0],
        "Reservoir_Porosity": group["Reservoir_Porosity"].iloc[0],
        "Reservoir_Permeability_mD": group["Reservoir_Permeability_mD"].iloc[0],
        "Lithology": group["Lithology"].iloc[0],
        "Log_Scaling": np.log(group["Target_Scaling_Factor"].iloc[0])
    }
    
    for i in range(len(core_pc_norm)):
        record[f"Pc_norm_{i}"] = core_pc_norm[i]
        
    records.append(record)

X_df = pd.DataFrame(records)

# Prepare Target and Features
y = X_df["Log_Scaling"]
X = X_df.drop(columns=["Curve_ID", "Log_Scaling"])

cat_features = ["Lithology"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

cat_model = CatBoostRegressor(
    iterations=1000, 
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE", 
    cat_features=cat_features,
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

# Evaluation
cat_pred_log = cat_model.predict(X_test)
cat_true = np.exp(y_test)
cat_pred = np.exp(cat_pred_log)

mae = mean_absolute_error(cat_true, cat_pred)
rmse = np.sqrt(mean_squared_error(cat_true, cat_pred))
r2 = r2_score(cat_true, cat_pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

# Plot Actual vs Predicted
plt.figure(figsize=(7,7))
plt.scatter(cat_true, cat_pred, alpha=0.5)
xmin, xmax = cat_true.min(), cat_true.max()
plt.plot([xmin, xmax], [xmin, xmax], "r--", linewidth=2)
plt.xlabel("True Scaling Factor")
plt.ylabel("Predicted Scaling Factor")
plt.title("CatBoost Predictions")
plt.grid(True)
plt.show()

# Save Model
cat_model.save_model("ScalingFactorCatBoost4.cbm")

0:	learn: 1.1166253	test: 1.1284628	best: 1.1284628 (0)	total: 251ms	remaining: 4m 10s
100:	learn: 0.2093386	test: 0.2165435	best: 0.2165435 (100)	total: 9.48s	remaining: 1m 24s
200:	learn: 0.1106206	test: 0.1166481	best: 0.1166481 (200)	total: 18.5s	remaining: 1m 13s
300:	learn: 0.0842363	test: 0.0895785	best: 0.0895785 (300)	total: 27.8s	remaining: 1m 4s
400:	learn: 0.0708057	test: 0.0777429	best: 0.0777429 (400)	total: 37.1s	remaining: 55.5s
500:	learn: 0.0640141	test: 0.0737167	best: 0.0737167 (500)	total: 46.7s	remaining: 46.5s


In [ ]:
# ---------------------------------------------------------
# Plot Actual vs Predicted Upscaled Curve for a Single Sample
# ---------------------------------------------------------

# 1. Select a random sample from the test set (e.g., the 0th index)
sample_iloc = 0
original_idx = X_test.index[sample_iloc]

# 2. Get the Curve_ID to fetch the original un-normalized data
sample_curve_id = X_df.loc[original_idx, "Curve_ID"]
lithology = X_test.iloc[sample_iloc]["Lithology"]

# 3. Retrieve the original Sw and Core_Pc from the long dataframe
sample_data = df[df["Curve_ID"] == sample_curve_id].sort_values("Sw")
sw_values = sample_data["Sw"].values
core_pc_values = sample_data["Core_Pc"].values

# 4. Get the True and Predicted Scaling Factors for this specific sample
true_scale = cat_true.iloc[sample_iloc]
pred_scale = cat_pred[sample_iloc]

# 5. Calculate the Upscaled Curves
true_upscaled_pc = core_pc_values * true_scale
pred_upscaled_pc = core_pc_values * pred_scale

# 6. Plotting
plt.figure(figsize=(9, 6))

# Plot Original Core Curve
plt.plot(sw_values, core_pc_values, 'k-', linewidth=2, alpha=0.5, label='Original Core $P_c$ (Input)')

# Plot True Reservoir Curve (Ground Truth)
plt.plot(sw_values, true_upscaled_pc, 'b-', linewidth=3, alpha=0.7, label=f'True Upscaled $P_c$ (Scale: {true_scale:.3f})')

# Plot Predicted Reservoir Curve (ML Prediction)
plt.plot(sw_values, pred_upscaled_pc, 'r--', linewidth=2.5, label=f'Predicted Upscaled $P_c$ (Scale: {pred_scale:.3f})')

# Formatting
plt.title(f'Capillary Pressure Curve Comparison\n(Curve ID: {sample_curve_id} | Lithology: {lithology})', fontsize=13)
plt.xlabel('Water Saturation ($S_w$, fraction)', fontsize=11)
plt.ylabel('Capillary Pressure ($P_c$)', fontsize=11)

# Industry standard is to plot Capillary Pressure on a logarithmic scale
plt.yscale('log')
plt.legend(loc='upper right')
plt.grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()